# RAG-Powered Document Q&A — Interactive Walkthrough

This notebook is a **step-by-step companion** to `rag.py`. It walks through a complete Retrieval-Augmented Generation (RAG) pipeline over local PDF research papers, one cell at a time, so you can:

- Run each stage in isolation and inspect the output.
- Tweak parameters (chunk size, `k`, prompt) and immediately see the effect.
- Understand **why** each step exists — not just what it does.

## What you'll build

```
PDFs → 1. Load → 2. Chunk → 3. Embed & Store → 4. Retrieve → 5. Generate → 6. Evaluate
```

| Step | Tool | Purpose |
|---|---|---|
| 1. Load | `PyPDFDirectoryLoader` | Read PDFs into LangChain `Document`s (one per page). |
| 2. Chunk | `RecursiveCharacterTextSplitter` | Split pages into ~1000-char chunks with overlap. |
| 3. Embed & Store | `OpenAIEmbeddings` + `Chroma` | Vectorize chunks and persist them locally. |
| 4. Retrieve | `vs.similarity_search` | Pull the top-k most relevant chunks per query. |
| 5. Generate | `RetrievalQA` + `gpt-4o-mini` | Produce a grounded answer with inline citations. |
| 6. Evaluate | `eval_set.json` | Score retrieval / faithfulness / correctness on a mini test set. |

## Prerequisites

1. Python 3.11+ (developed on 3.14).
2. An **OpenAI API key** in a root `.env` file:
   ```env
   OPENAI_API_KEY=sk-...
   ```
3. Dependencies installed inside the project's virtual environment:
   ```powershell
   python -m venv .venv
   .\.venv\Scripts\Activate.ps1
   pip install -r requirements.txt
   ```
4. Source PDFs present under `Documents/`.

> **Tip:** select the `.venv` kernel in your notebook before running the cells below.

## Setup — imports & configuration

We import everything up front and pull the OpenAI API key from `.env` via `python-dotenv`. The constants below mirror `rag.py`; feel free to tweak them later in the notebook to experiment.

In [ ]:
import json
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

load_dotenv()

DOCS_DIR = "Documents"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
EMBED_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"
PERSIST_DIR = "chroma_db"
COLLECTION = "papers"
EVAL_FILE = "eval_set.json"
# k=6 because top-4 sometimes pulled title/references pages and starved the LLM
# of body content, producing "I don't know" on otherwise-answerable questions.
TOP_K = 6

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set. Create a .env file from .env.example."
print("OpenAI key loaded.")
print(f"Docs dir       : {DOCS_DIR}")
print(f"Embedding model: {EMBED_MODEL}")
print(f"LLM            : {LLM_MODEL}")
print(f"Vector store   : ChromaDB at ./{PERSIST_DIR} (collection='{COLLECTION}')")

## Step 1 — Load the PDFs

We use **`PyPDFDirectoryLoader`** from `langchain-community` to read every PDF in the `Documents/` folder. It's backed by `pypdf` and returns a list of LangChain `Document` objects.

> Important: this loader yields **one `Document` per page**, not per file. That's why our 6 PDFs produce 49 `Document`s. Each `Document` carries `metadata` with its `source` (file path) and `page` number — we'll rely on both later for citations.

In [ ]:
loader = PyPDFDirectoryLoader(DOCS_DIR)
docs = loader.load()

unique_files = sorted({os.path.basename(d.metadata["source"]) for d in docs})
print(f"Number of PDF files              : {len(unique_files)}")
print(f"Number of pages (Documents)      : {len(docs)}")
print("\nFiles found:")
for f in unique_files:
    print(f"  - {f}")

print("\nPreview of Document 0:")
print(f"  source : {os.path.basename(docs[0].metadata.get('source', ''))}")
print(f"  page   : {docs[0].metadata.get('page')}")
print(f"  content: {docs[0].page_content[:300].strip()!r}...")

## Step 2 — Chunk the pages

Pages are typically too large to embed as a single unit, and they often mix multiple topics. We use **`RecursiveCharacterTextSplitter`** to break them into smaller, retrieval-friendly chunks.

- `chunk_size = 1000` characters (not tokens) — a reasonable middle ground.
- `chunk_overlap = 150` — overlap keeps sentences that span chunk boundaries retrievable from either side.

The splitter tries a ranked list of separators (`\n\n`, `\n`, `" "`, …) so it splits on *semantic* boundaries first, falling back to smaller ones only when needed.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(docs)

sizes = [len(c.page_content) for c in chunks]
print(f"Total chunks created : {len(chunks)}")
print(f"Smallest chunk       : {min(sizes)} chars")
print(f"Largest chunk        : {max(sizes)} chars")
print(f"Mean chunk size      : {sum(sizes) // len(sizes)} chars")

print("\nSample chunk (index 0):")
print(f"  source : {os.path.basename(chunks[0].metadata.get('source', ''))}")
print(f"  page   : {chunks[0].metadata.get('page')}")
print(f"  content: {chunks[0].page_content[:250].strip()!r}...")

## Step 3 — Embed and store in ChromaDB

Now we turn each chunk into a **dense vector** using OpenAI's `text-embedding-3-small` (1536 dimensions — good quality/cost balance) and persist the vectors into a local **ChromaDB** collection on disk.

**Caching behavior**

- On **first run**: all 244 chunks are embedded (one API call batch) and written to `./chroma_db/`.
- On **subsequent runs**: the existing collection is reused — no re-embedding, no API cost.
- **To force a rebuild** (e.g. after changing `CHUNK_SIZE` or the embedding model): delete the `chroma_db/` folder.

| Component | Choice | Why |
|---|---|---|
| Embedding | `text-embedding-3-small` | ~$0.02 per 1M tokens, 1536-dim, strong recall. |
| Vector DB | ChromaDB (local) | Zero-setup, persists to disk, perfect for a single-script/notebook project. |

In [ ]:
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
vs = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR,
)

if vs._collection.count() == 0:
    print(f"Empty collection — embedding {len(chunks)} chunks (this calls the OpenAI API)...")
    vs.add_documents(chunks)
    print("Done.")
else:
    print("Existing collection detected — reusing cached vectors (no API calls).")

print(f"\nVectors stored in '{COLLECTION}': {vs._collection.count()}")
print(f"Persistence directory         : ./{PERSIST_DIR}")

## Step 4 — Test retrieval *before* wiring up the LLM

This is the single biggest quality lever in a RAG system: **if retrieval is bad, the LLM cannot recover.** Before spending tokens on generation, we verify the retriever actually surfaces the right chunks.

We hand-picked three queries that target distinct papers in the corpus:

| # | Query | Targeted paper(s) |
|---|---|---|
| 1 | *"How is electric power transmission network topology processed efficiently?"* | Topology Processing paper |
| 2 | *"How are graph models of power networks constructed from synchrophasor PMU data?"* | Data-Driven Graph Construction + Graph Models Using Synchrophasor Data |
| 3 | *"How does an LVQ neural network identify power system network branch events online?"* | LVQ Neural Network paper |

For each, we pull the top `k = 3` chunks and inspect them. A common pattern you'll notice: title/references pages can rank highly because they're keyword-dense. We address that in Step 5 by bumping `k` for the LLM.

In [ ]:
test_queries = [
    "How is electric power transmission network topology processed efficiently?",
    "How are graph models of power networks constructed from synchrophasor PMU data?",
    "How does an LVQ neural network identify power system network branch events online?",
]

for q in test_queries:
    print("\n" + "=" * 80)
    print(f"QUERY: {q}")
    print("=" * 80)
    hits = vs.similarity_search(q, k=3)
    for i, h in enumerate(hits, 1):
        src = os.path.basename(h.metadata.get("source", ""))
        page = h.metadata.get("page")
        preview = h.page_content.strip().replace("\n", " ")[:300]
        print(f"\n[{i}] {src} (page {page})")
        print(f"    {preview}...")

## Step 5 — Build the RAG chain

Now we wire the retriever into `gpt-4o-mini` with a carefully constrained prompt:

- **`RetrievalQA.from_chain_type(chain_type="stuff")`** — the simplest chain: concatenate all retrieved chunks into the prompt.
- **`temperature=0`** — deterministic, faithful answers.
- **`k = 6`** — originally `k = 4`, but the top-4 for Q1 were the title page and a references page. Bumping to `k = 6` pulled in body-content chunks and fixed a spurious *"I don't know"*. This is the exact pattern Step 4 warned us about.
- **Custom prompt** — forbids invention, requires an explicit refusal phrase, and asks for inline citations like `[source: <filename> p.<page>]`.
- **`document_prompt`** — prefixes each retrieved chunk with its `source` and `page` *before* it reaches the LLM. Without this, inline citations come out as literal `<filename>` placeholders.

In [ ]:
PROMPT_TEMPLATE = """You are a helpful research assistant answering questions about
electric power transmission network research papers.

Use ONLY the context below to answer the question. If the context does not contain
the answer, say "I don't know based on the provided documents." Do not invent facts.
Be concise and technical. Cite sources inline as [source: <filename> p.<page>].

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

document_prompt = PromptTemplate(
    template="[source: {source} p.{page}]\n{page_content}",
    input_variables=["source", "page", "page_content"],
)

llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vs.as_retriever(search_kwargs={"k": TOP_K}),
    chain_type_kwargs={"prompt": prompt, "document_prompt": document_prompt},
    return_source_documents=True,
)

print(f"RAG chain ready — LLM={LLM_MODEL}, k={TOP_K}, temperature=0")

### Ask the three test questions

Let's send each of our Step-4 queries through the full chain and inspect both the generated answer and the source documents the retriever returned. Each answer should:

1. Be grounded in the retrieved chunks (no hallucinations).
2. Include inline `[source: ... p.X]` citations.
3. Reference the correct paper(s) in its sources list.

In [ ]:
for q in test_queries:
    print("\n" + "=" * 80)
    print(f"QUESTION: {q}")
    print("=" * 80)
    result = qa_chain.invoke({"query": q})
    print(f"\nANSWER:\n{result['result']}")
    print("\nSOURCES:")
    seen = set()
    for d in result["source_documents"]:
        key = (os.path.basename(d.metadata.get("source", "")),
               d.metadata.get("page"))
        if key not in seen:
            seen.add(key)
            print(f"  - {key[0]} (page {key[1]})")

## Step 6 — Evaluate with a mini eval set

A "looks good" eyeball test isn't enough. We score the pipeline along three axes:

| Axis | What it measures |
|---|---|
| **Retrieval** ✅ | Did we pull at least one of the expected source files? (Waived for refusal questions.) |
| **Faithfulness** ✅ | Did the answer cite a retrieved file inline and avoid hallucinating? For refusal questions, the model must correctly refuse. |
| **Correctness** ✅ | Do all expected keywords appear in the answer? For the refusal question, the answer must contain *"I don't know"*. |

The eval set (`eval_set.json`) has **5 items**. Item #5 is a deliberate **refusal test** — a topic the corpus doesn't cover — to verify the model declines instead of making things up.

| # | Question (abridged) | Targeted paper | Type |
|---|---|---|---|
| 1 | PMU sampling rate (Hz)? | Data-Driven Graph Construction | factual |
| 2 | Test system used on the RTDS? | Data-Driven Graph Construction | factual |
| 3 | Neural net + IEEE benchmark for branch events? | LVQ Neural Network | factual |
| 4 | Distributed architecture for event ID? | LVQ / Distributed Identification | factual |
| 5 | Do the papers propose deep reinforcement learning? | — (not covered) | **refusal** |

In [ ]:
with open(EVAL_FILE, "r", encoding="utf-8") as f:
    eval_items = json.load(f)

print(f"Loaded {len(eval_items)} eval items from {EVAL_FILE}")
print("\nPreview:")
for it in eval_items:
    marker = "  (refusal test)" if it.get("refusal_expected") else ""
    print(f"  Q{it['id']}: {it['question']}{marker}")

### Auto-grading helper

The grader below applies the three rules from the rubric. Note: this is a **coarse, cheap filter** — keyword overlap and filename presence. It catches regressions well but is not a substitute for manual review.

In [ ]:
def grade_item(item, result):
    """Auto-grade one eval item. Manual review is still recommended."""
    answer = result["result"]
    answer_lc = answer.lower()
    retrieved_files = {
        os.path.basename(d.metadata.get("source", ""))
        for d in result["source_documents"]
    }
    refused = "i don't know" in answer_lc

    if item["refusal_expected"]:
        retrieval_ok = True  # N/A for refusal items
        faithfulness_ok = refused
        correctness_ok = refused
    else:
        retrieval_ok = any(s in retrieved_files for s in item["expected_sources"])
        cited_retrieved = any(s in answer for s in retrieved_files)
        faithfulness_ok = (not refused) and cited_retrieved
        correctness_ok = all(kw in answer_lc for kw in item["expected_keywords"])

    return {
        "retrieval": retrieval_ok,
        "faithfulness": faithfulness_ok,
        "correctness": correctness_ok,
        "answer": answer,
        "retrieved_files": sorted(retrieved_files),
    }

### Run the eval & print the scorecard

In [ ]:
tick = lambda b: "PASS" if b else "FAIL"
rows = []

for item in eval_items:
    result = qa_chain.invoke({"query": item["question"]})
    g = grade_item(item, result)
    rows.append((item, g))
    print("\n" + "-" * 80)
    print(f"Q{item['id']}: {item['question']}")
    print(f"Expected : {item['expected_answer']}")
    print(f"Got      : {g['answer']}")
    print(f"Retrieved: {g['retrieved_files']}")
    print(f"  Retrieval   : {tick(g['retrieval'])}")
    print(f"  Faithfulness: {tick(g['faithfulness'])}")
    print(f"  Correctness : {tick(g['correctness'])}")

n = len(rows)
r = sum(1 for _, g in rows if g["retrieval"])
f_ = sum(1 for _, g in rows if g["faithfulness"])
c = sum(1 for _, g in rows if g["correctness"])

print("\n" + "=" * 80)
print(f"SCORECARD  (n={n})")
print("=" * 80)
print(f"  Retrieval   : {r}/{n}")
print(f"  Faithfulness: {f_}/{n}")
print(f"  Correctness : {c}/{n}")
print("\n(Auto-graded — please double-check answers manually against the PDFs.)")

## Try your own question

Now it's your turn. Edit the `my_question` string below and re-run the cell. The answer must come from the 6 papers in `Documents/` — ask the model something you know is in there, or probe its refusal behavior by asking about a topic that isn't.

In [ ]:
my_question = "What role do PMUs play in monitoring electric power transmission networks?"

result = qa_chain.invoke({"query": my_question})
print(f"QUESTION: {my_question}\n")
print(f"ANSWER:\n{result['result']}\n")
print("SOURCES:")
seen = set()
for d in result["source_documents"]:
    key = (os.path.basename(d.metadata.get("source", "")), d.metadata.get("page"))
    if key not in seen:
        seen.add(key)
        print(f"  - {key[0]} (page {key[1]})")

## Where to go next

You now have a working RAG pipeline end-to-end. Good follow-up experiments:

- **Chunking:** try `chunk_size = 500` or `2000` and compare retrieval quality. Remember to delete `chroma_db/` to force re-embedding.
- **`k`:** set `TOP_K = 3` and see which questions break; set `TOP_K = 10` and see if the LLM gets distracted by noisy context.
- **Prompt:** make it stricter ("quote verbatim") or looser ("synthesize across sources") and measure the effect on faithfulness vs. correctness.
- **Embedding model:** swap to `text-embedding-3-large` (3072-dim) — higher quality, higher cost. Delete `chroma_db/` first.
- **Eval set:** add multi-hop questions (spanning 2+ papers) and adversarial prompts (ambiguous wording, wrong-paper distractors).
- **Drop references pages:** preprocess docs by cutting content after the `"REFERENCES"` heading and see whether retrieval precision rises.
- **LLM-as-judge:** replace the keyword grader with a second LLM call that scores each answer against the retrieved chunks — a much stronger faithfulness signal.

Happy retrieving!